### Importing Libraries
 
Import necessary Libraries utilised for data loading, reading, plotting, coarse graining or averaging and for other computations

In [1]:
import gzip
import re
import os
import glob
import torch
import numpy as np
import matplotlib
from pathlib import Path
from scipy.spatial import cKDTree
from scipy.special import erf
import matplotlib.pyplot as plt
import torch.nn.functional as F
from multiprocessing import cpu_count
from typing import Tuple, List, Optional
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter

### DEM Simulation Data File Reading

In [2]:
BULK_TYPE = 3

class FileReader:
    """ 
    =============================================================================================================================
    Utility class which contains all the functions related to reading and storing the DEM dump* files data in proper order for:
        1. Stress
        2. Velocity, used for computing Deformation rate tensor
        3. Position, used for computing local variation of solid volume fraction: phi. 

    The DEM output file should contain the following quantities in the dump file in the following order:
    'id type x y z vx vy vz c_s_total[1] c_s_total[2] c_s_total[3] c_s_total[4] c_s_total[5] c_s_total[6]'
    - id: particle ID (count of particles)
    - type: Partucle type (bulk, top or bottom wall)
    - x,y,z: particle positions
    - vx, vy, vz: particle velocities
    - c_s_total[1-6]: 6 independent components of the stress tensor (σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz) for each particle, multiplied by the particle volume (V_i) as per LAMMPS output convention.
    =============================================================================================================================
    """

    def get_avg_coordNum(self, file_path):
        """Get average coordination number from the output file."""
        last_value = None

        with open(file_path, 'r') as f:
            for line in f:
                # Check if line starts with a number (timestep line)
                if line.strip() and line.split()[0].isdigit():
                    # Get the last column value
                    columns = line.split()
                    last_value = columns[-1]
        coordination_number = float(last_value)
        return coordination_number

    def read_radius_from_config(self, config_file):
        """
        Read LAMMPS config file and extract radius from diameter column
        Returns array of radius values ordered by atom ID
        """
        with open(config_file, 'r') as f:
            lines = f.readlines()

        # Find Atoms section
        particles_start = None
        for i, line in enumerate(lines):
            if 'Atoms' in line:
                particles_start = i + 2  # Skip "Atoms" line and blank line
                break
            
        if particles_start is None:
            raise ValueError("Atoms section not found in config file")

        # Extract ID and diameter
        particle_data = []
        for line in lines[particles_start:]:
            if not line.strip():  # Stop at blank line
                break
            parts = line.split()
            if len(parts) >= 3:
                particle_id = int(parts[0])
                diameter = float(parts[2])  # dia is 3rd column
                particle_data.append((particle_id, diameter))

        # Sort by particle ID to ensure correct order
        particle_data.sort(key=lambda x: x[0])

        # Extract diameters and convert to radius
        diameters = np.array([d for _, d in particle_data])
        radius = diameters / 2.0

        return radius

    def read_dump_file(self, file_path):
        """
        Read a LAMMPS dump file and extract particle data
        Returns a dictionary with keys: 'N', 'dim', 'x', 'v', 'type', etc.
        """
        with open(file_path, 'r') as f:
            lines = f.readlines()

        frame_data = {}
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            if line.startswith("ITEM: TIMESTEP"):
                i += 1
                frame_data['timestep'] = int(lines[i].strip())
            elif line.startswith("ITEM: NUMBER OF ATOMS"):
                i += 1
                frame_data['N'] = int(lines[i].strip())
            elif line.startswith("ITEM: BOX BOUNDS"):
                i += 1
                bounds = []
                for _ in range(3):
                    bounds.append(list(map(float, lines[i].strip().split())))
                    i += 1
                frame_data['box_bounds'] = np.array(bounds)
                frame_data['dim'] = 3  # Assuming 3D
                continue  # Skip incrementing i here
            elif line.startswith("ITEM: ATOMS"):
                headers = line.split()[2:]  # Get column headers
                data = []
                for j in range(frame_data['N']):
                    i += 1
                    parts = lines[i].strip().split()
                    data.append([float(part) for part in parts])
                data_array = np.array(data)

                # Map headers to data columns
                for idx, header in enumerate(headers):
                    frame_data[header] = data_array[:, idx]
            i += 1

        return frame_data

    def read_all_frames(self, folder_path, pattern="dump.stress.*", max_frames=1000000000):
        """
        Read all dump files in the specified folder matching the pattern
        Returns a list of frames (dictionaries)
        """
        file_paths = sorted(glob.glob(os.path.join(folder_path, pattern)))
        frames = []
        print(f"Reading from {folder_path}")
        for file_path in file_paths[:max_frames]:
            frame = self.read_dump_file(file_path)
            frames.append(frame)
        print(f"Done! total Frames: {len(frames)} dump files")
        return frames

### Coarse Graining Class
** All functions used for coarse graining are defined below

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[CG] Using device: {DEVICE}")

class CompleteCoarseGraining:
    """
    ============================================================================
    -- COARSE GRAINING METHODS for DEM SIMULATION DATA (Spatial and Temporal) --
    ============================================================================

    1. STRESS:
        Irving-Kirkwood-Hardy (IKH) coarse-graining of per-particle stress tensors.

        CG formula:
        ----------
            σ(r) = Σ_i [ σ_i · φ(|r−r_i|, ξ) ]
                   ─────────────────────────────────────────
                   Z(y_r) · Σ_i [ V_i · φ(|r−r_i|, ξ) ]

        where:
            σ_i    = per-particle stress (stress × volume) from LAMMPS
            V_i    = particle volume  (4π R_i³ / 3)
            φ      = 3-D normalised Gaussian kernel, width ξ
            Z(y_r) = erf-based renormalisation for boundary truncation (see below)

        Kernel width ξ:
        --------------
        Two operating modes, set via the `xi_mode` argument of
        `estimate_coarse_graining_width`:

            'particle'  (default, IKH near-wall profile mode)
                ξ = 1·dp = 2R̄  — resolves layering oscillations at 1 dp
                Truncation correction is essential for first ~2 nodes near wall.
                Wall nodes (n=0, n=N-1) are skipped automatically.

            'smooth'    (bulk momentum-balance diagnostic mode)
                ξ = 3·dp = 6R̄  — suppresses force-chain noise enough that
                ∇·σ diagnostics are reliable.  Truncation matters for first ~6
                nodes near wall.

        Boundary normalisation Z(y):
        ---------------------------
        For a Gaussian kernel centred at grid node y_n, the accessible-domain
        kernel integral is:

            Z(y_n) = ∫_{y_lo}^{y_hi} φ(y − y_n ; ξ) dy
                   = 0.5 · [ erf((y_n − y_lo)/(√2·ξ)) + erf((y_hi − y_n)/(√2·ξ)) ]

        This is exactly 1 in the bulk (node > ~3ξ from either wall) and < 1 near
        the walls.  Dividing the denominator by Z(y_n) restores correct
        normalisation without adding any non-physical particles or forces.

        Note: ghost/mirror particles are NOT used.  At particle-scale resolution
        (ξ ≈ 1 dp) ghost particles would double-count near-wall particles already
        within the support and introduce artefacts.  The erf renormalisation is the
        analytically exact correction for a Gaussian kernel with a flat wall.

        Wall particle exclusion:
        -----------------------
        Wall particles (type != BULK_TYPE, i.e. the constrained boundary atoms)
        are excluded from all summations.  Their wall-fluid contact stress is
        already captured in the LAMMPS virial of the bulk particle on the other
        side of each contact, so no force contribution is lost.

        Grid convention:
        ---------------
        The y-grid spans [y_lo + Δy, y_hi − Δy] — the first and last fluid
        layers, skipping the wall-node positions themselves.  With Δy = 1 dp
        this gives nodes n = 1 … N-1 (0-indexed), consistent with the IKH
        recommendation to skip n=0 and n=N.
    
    2. VELOCITY (DEFORMATION RATE):
        Goldhirsch-Weinhart coarse-graining implementation for DEM Simulation data.
    
        Velocities are computed via finite differences of particle positions between
        consecutive dump frames:
            v_i = (x_{i+1} - x_i) / delta_t

        The first frame (index 0) is assigned zero velocity (initially at rest).
        If 1001 frames are read, 1000 FD velocities are computed; frame 0 gets v=0,
        frames 1..1000 get the FD velocity computed from the preceding pair.

    3. PHI (local variation of Solif Volume Fraction):
    Two-stage local solid volume fraction for DEM granular data.

        Gridding
        --------
          Full box  : Ly = 0.40 m  (Ny_full = 40 at d = 0.01 m)
          Bulk range: y ∈ [y_lo+d, y_hi−d]  →  Ly_eff = 0.38 m
          Grid      : Nx=50, Ny=38, Nz=10
          PBC       : x (flow), z (vorticity)  →  circular padding
          Walls     : y (gradient)             →  reflect padding

    *. Device strategy
    ---------------
      cKDTree        : always CPU (scipy limitation — used only for candidate lookup)
      tensor math    : GPU if available, else CPU
      Rule           : every tensor is created/moved via the module-level DEVICE;
                       .cpu().numpy() is called only when passing to cKDTree.
    """

    def __init__(self):
        """
        Parameters
        ----------
        folder_path : str | Path
        pattern : str
            Glob pattern for dump files.
        max_frames : int | float
            Maximum number of frames to load.
        n_cores : int | None
            Number of CPU cores for parallel work.
        dt : float
            Physical time represented by one LAMMPS timestep (seconds).
            delta_t between two frames = (timestep_j - timestep_i) * dt.
        timestep_to_seconds : float
            Alias for dt; if both are given dt takes precedence.
            Kept for backwards compatibility.
        """
        self.dt = 5.18e-6                          # physical seconds per LAMMPS timestep
 
        self.type_density = {1: 2600.0, 2: 2600.0, 3: 2600.0}   # kg/m³
 
        self.w = None
        self.support_fac = 3.0
        self.support = None
        self.dx = None


    def gaussian_kernel(self, r, w):
        """Normalised 3-D Gaussian φ(r) = (1/√(2π)w)³ exp(−r²/2w²)."""
        norm = 1.0 / ((np.sqrt(2.0 * np.pi) * w) ** 3)
        return norm * np.exp(-r ** 2 / (2.0 * w ** 2))
 
    def boundary_normalization(self, y_pts, y_lo, y_hi, w):
        """
        Fraction of 1-D Gaussian mass inside [y_lo, y_hi] (the true wall positions).
 
            Z(y) = 0.5 · [erf((y − y_lo)/(√2·w)) + erf((y_hi − y)/(√2·w))]
 
        Z = 1 in bulk;  Z < 1 near walls.  Pass actual wall coords, not ±w offsets.
        """
        sq2w = np.sqrt(2.0) * w
        Z = 0.5 * (erf((y_pts - y_lo) / sq2w) + erf((y_hi - y_pts) / sq2w))
        return np.clip(Z, 1e-6, 1.0)
 
    def compute_particle_volume(self, frame):
        return (4.0 / 3.0) * np.pi * frame['radius'] ** 3
 
    def compute_particle_mass(self, frame):
        """Per-particle mass from type-density table and particle radius."""
        density = np.array([self.type_density.get(int(t), 2600.0) for t in frame['type']])
        return density * (4.0 / 3.0) * np.pi * frame['radius'] ** 3
 
    def stress_voigt_to_tensor(self, stress_voigt):
        """[σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz] → (N,3,3)."""
        T = np.zeros((len(stress_voigt), 3, 3))
        T[:, 0, 0] = stress_voigt[:, 0]
        T[:, 1, 1] = stress_voigt[:, 1]
        T[:, 2, 2] = stress_voigt[:, 2]
        T[:, 0, 1] = T[:, 1, 0] = stress_voigt[:, 3]
        T[:, 0, 2] = T[:, 2, 0] = stress_voigt[:, 4]
        T[:, 1, 2] = T[:, 2, 1] = stress_voigt[:, 5]
        return T
 
    def estimate_coarse_graining_width(self, frame, xi_mode='particle'):
        """
        CG kernel width ξ.
          'particle' → ξ = 2·dp  (resolves layering; use for near-wall profiles)
          'smooth'   → ξ = 3·dp  (suppresses force-chain noise; use for ∇·σ)
        """
        mean_r = np.mean(frame['radius'])
        dp = 2.0 * mean_r
 
        if xi_mode == 'particle':
            w, label = 2.0 * dp, "2·dp (particle-scale IKH)"
        elif xi_mode == 'smooth':
            w, label = 3.0 * dp, "3·dp (smooth, momentum-balance)"
        else:
            raise ValueError(f"xi_mode must be 'particle' or 'smooth', got '{xi_mode}'")
 
        print(f"[CG width]  R̄={mean_r:.6f} m  dp={dp:.6f} m  ξ={label}={w:.6f} m  "
              f"support={self.support_fac}·ξ={self.support_fac*w:.6f} m")
        return w
 
    def create_grid(self, frame, n_points=None, dx=None, y_margin=None, xi_mode='particle'):
        """
        Uniform evaluation grid.  Default: Δ = dp, margin = dp from each y-wall.
        erf renormalisation handles all near-wall nodes — no node is excluded.
        """
        bounds = frame['box_bounds'].copy()
        dim = frame['dim']
 
        if self.w is None:
            self.w = self.estimate_coarse_graining_width(frame, xi_mode=xi_mode)
        w = self.w
        dp = 2.0 * np.mean(frame['radius'])
 
        if y_margin is None:
            y_margin = dp
 
        bounds[1, 0] += y_margin
        bounds[1, 1] -= y_margin
 
        if n_points is None and dx is None:
            dx = dp
            print(f"[grid]  auto Δ = dp = {dx:.6f} m")
 
        if n_points is None:
            dx_arr = np.full(dim, dx) if np.isscalar(dx) else np.asarray(dx)
            n_points = tuple(
                max(2, int(round((bounds[i, 1] - bounds[i, 0]) / dx_arr[i])) + 1)
                for i in range(dim)
            )
 
        if isinstance(n_points, int):
            n_points = (n_points,) * dim
 
        axes = [np.linspace(bounds[i, 0], bounds[i, 1], n_points[i]) for i in range(dim)]
        grids = np.meshgrid(*axes, indexing='ij')
        grid_points = np.column_stack([g.ravel() for g in grids])
 
        actual_dy = (bounds[1, 1] - bounds[1, 0]) / (n_points[1] - 1) if n_points[1] > 1 else 1.0
        nyquist_ok = actual_dy <= w / 2.0
        print(f"[grid]  shape={n_points}  y=[{bounds[1,0]:.4f},{bounds[1,1]:.4f}]  "
              f"Δy={actual_dy:.4f}  ξ/2={w/2:.4f}  "
              f"{'✓ Nyquist OK' if nyquist_ok else '⚠ Δy > ξ/2'}")
        return grid_points, n_points
 
    # ─────────────────────────────────────────────────────────────────────────
    # 1. STRESS  (IKH)
    # ─────────────────────────────────────────────────────────────────────────
 
    def coarse_grain_stress(self, frame, grid_points, w=None, xi_mode='particle'):
        """
        IKH coarse-grain per-particle stress → continuum field.
        PBC: x, z periodic.  y wall-bounded (erf renormalisation, no ghost particles).
        Wall particles (type ≠ BULK_TYPE) excluded from summations.
 
        Returns
        -------
        stress_field : (N_grid, 3, 3)  [Pa]
        """
        if w is None:
            w = self.estimate_coarse_graining_width(frame, xi_mode=xi_mode)
        self.w = w
 
        support = self.support_fac * w
 
        pos = frame['x'][:, :3]
        sigma_p = self.stress_voigt_to_tensor(frame['stress'])
        vol = self.compute_particle_volume(frame)
 
        if np.sum(np.abs(frame['stress'])) < 1e-12:
            print("WARNING: all stress values near-zero — check LAMMPS compute")
 
        # Box geometry
        x_lo = frame['box_bounds'][0, 0];  Lx = frame['box_bounds'][0, 1] - x_lo
        wall_y_lo = frame['box_bounds'][1, 0]             # FIX: actual wall position
        wall_y_hi = frame['box_bounds'][1, 1]             # FIX: actual wall position
        z_lo = frame['box_bounds'][2, 0];  Lz = frame['box_bounds'][2, 1] - z_lo
 
        # Shift x, z into [0, L) for cKDTree PBC; y stays absolute
        pos_s = pos.copy()
        pos_s[:, 0] = (pos[:, 0] - x_lo) % Lx
        pos_s[:, 2] = (pos[:, 2] - z_lo) % Lz
 
        tree = cKDTree(pos_s, boxsize=[Lx, 1e30, Lz])
 
        n_grid = len(grid_points)
        stress_field = np.zeros((n_grid, 3, 3))
        has_data = np.zeros(n_grid, dtype=bool)
 
        # erf renormalisation using actual wall positions (not ±w offsets)
        Z = self.boundary_normalization(grid_points[:, 1], wall_y_lo, wall_y_hi, w)
        n_corrected = np.sum(Z < 0.99)
        print(f"[stress CG]  {n_grid} pts  ξ={w:.5f} m  nodes with Z<0.99: {n_corrected}")
 
        for k, gpt in enumerate(grid_points):
            if k % 2000 == 0:
                print(f"  {k}/{n_grid}")
 
            gpt_s = gpt.copy()
            gpt_s[0] = (gpt[0] - x_lo) % Lx
            gpt_s[2] = (gpt[2] - z_lo) % Lz
 
            nbrs = tree.query_ball_point(gpt_s, support)
            if not nbrs:
                continue
 
            disp = pos_s[nbrs] - gpt_s
            disp[:, 0] -= Lx * np.round(disp[:, 0] / Lx)
            disp[:, 2] -= Lz * np.round(disp[:, 2] / Lz)
 
            r = np.linalg.norm(disp, axis=1)
            phi = self.gaussian_kernel(r, w)
 
            numerator = np.einsum('i,ijk->jk', phi, sigma_p[nbrs])
            denominator = Z[k] * np.dot(vol[nbrs], phi)   # Z corrects wall truncation
 
            if denominator > 1e-30:
                stress_field[k] = numerator / denominator
                has_data[k] = True
 
        print(f"  done. pts with data: {has_data.sum()}/{n_grid}")
        return stress_field
 
    # ─────────────────────────────────────────────────────────────────────────
    # 2. VELOCITY  (Goldhirsch-Weinhart)
    # ─────────────────────────────────────────────────────────────────────────
 
    def compute_finite_difference_velocities(self, frames):
        """
        Per-particle velocities from finite differences of positions.
        frame[0]['v_fd'] = 0;  frame[i]['v_fd'] = (x_i − x_{i-1}) / Δt.
        Uses self.dt (physical seconds per LAMMPS timestep).   # FIX: was hardcoded
        PBC unwrapping applied in periodic directions.
        """
        print("\n[FD vel]  computing …")
        n_frames = len(frames)
        dim = frames[0]['dim']
 
        # Sort particles by ID for consistent ordering
        for frame in frames:
            if frame.get('id') is not None:
                order = np.argsort(frame['id'])
                for key in ('x', 'v', 'type', 'radius'):
                    if frame.get(key) is not None:
                        frame[key] = frame[key][order]
                if frame.get('mass') is not None:
                    frame['mass'] = frame['mass'][order]
                frame['id'] = frame['id'][order]
 
        frames[0]['v_fd'] = np.zeros((frames[0]['N'], dim))
 
        for i in range(1, n_frames):
            prev, curr = frames[i - 1], frames[i]
            delta_t = (curr['timestep'] - prev['timestep']) * self.dt  # FIX: self.dt
 
            if delta_t == 0:
                print(f"  ⚠ frame {i}: Δt=0, setting v_fd=0")
                curr['v_fd'] = np.zeros((curr['N'], dim))
                continue
 
            dx = curr['x'][:, :dim] - prev['x'][:, :dim]
 
            box_lengths = curr['box_bounds'][:dim, 1] - curr['box_bounds'][:dim, 0]
            for d in range(dim):
                btype = curr['boundary_types'][d] if d < len(curr['boundary_types']) else 'pp'
                if 'p' in btype:
                    dx[:, d] -= box_lengths[d] * np.round(dx[:, d] / box_lengths[d])
 
            curr['v_fd'] = dx / delta_t
 
            if i % 100 == 0 or i == n_frames - 1:
                vmax = np.max(np.linalg.norm(curr['v_fd'], axis=1))
                print(f"  frame {i}/{n_frames-1}  |v_fd|_max={vmax:.4g}")
 
        print(f"✓ FD velocities: {n_frames} frames ({n_frames-1} non-zero + 1 zero)")
 
    def coarse_grain_velocity(self, frame, grid_points, w=None, use_fd_velocity=True):
        """
        Goldhirsch-Weinhart mass-weighted CG velocity.
 
        Returns
        -------
        velocity_field : (N_grid, dim)
        density_field  : (N_grid,)
        """
        if w is None:
            w = self.estimate_coarse_graining_width(frame)
        self.w = w
        support = self.support_fac * w
        dim = frame['dim']
        positions = frame['x'][:, :dim]
 
        if use_fd_velocity:
            if 'v_fd' not in frame:
                raise KeyError("'v_fd' missing — call compute_finite_difference_velocities first.")
            velocities = frame['v_fd'][:, :dim]
            vel_label = "finite-difference"
        else:
            velocities = frame['v'][:, :dim]
            vel_label = "dump-file"
 
        print(f"[vel CG]  source={vel_label}")
        mass = self.compute_particle_mass(frame)           # FIX: was missing entirely
        tree = cKDTree(positions)
 
        n_grid = len(grid_points)
        velocity_field = np.zeros((n_grid, dim))
        density_field = np.zeros(n_grid)
 
        for i, grid_pt in enumerate(grid_points):
            if i % 1000 == 0:
                print(f"  {i}/{n_grid}")
            indices = tree.query_ball_point(grid_pt[:dim], support)
            if not indices:
                continue
            r_vec = positions[indices] - grid_pt[:dim]
            r = np.linalg.norm(r_vec, axis=1)
            weights = self.gaussian_kernel(r, w) * mass[indices]
            total_weight = weights.sum()
            if total_weight > 1e-12:
                velocity_field[i] = (weights[:, None] * velocities[indices]).sum(0) / total_weight
                density_field[i] = total_weight
 
        print("  ✓ velocity CG done")
        return velocity_field, density_field
 
    def compute_velocity_profile(self, frame, direction='y', velocity_component='x',
                                 n_bins=50, w=None, use_fd_velocity=True):
        """1-D velocity profile averaged in bins along `direction`."""
        dir_map = {'x': 0, 'y': 1, 'z': 2}
        dir_idx = dir_map[direction.lower()] if isinstance(direction, str) else direction
        vel_idx = dir_map[velocity_component.lower()] if isinstance(velocity_component, str) else velocity_component
 
        positions = frame['x'][:, dir_idx]
        if use_fd_velocity:
            if 'v_fd' not in frame:
                raise KeyError("'v_fd' missing — call compute_finite_difference_velocities first.")
            velocities = frame['v_fd'][:, vel_idx]
            vel_label = "finite-difference"
        else:
            velocities = frame['v'][:, vel_idx]
            vel_label = "dump-file"
 
        if w is None:
            w = self.estimate_coarse_graining_width(frame)
 
        bounds = frame['box_bounds'][dir_idx]
        bin_edges = np.linspace(bounds[0] + w, bounds[1] - w, n_bins + 1)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
 
        velocity_profile = np.zeros(n_bins)
        velocity_std = np.zeros(n_bins)
        particle_count = np.zeros(n_bins)
 
        for i, center in enumerate(bin_centers):
            distances = np.abs(positions - center)
            mask = distances < (self.support_fac * w)
            if mask.sum() > 0:
                weights = self.gaussian_kernel(distances[mask], w)
                velocity_profile[i] = np.average(velocities[mask], weights=weights)
                velocity_std[i] = np.std(velocities[mask])
                particle_count[i] = mask.sum()
 
        print(f"[vel profile]  source={vel_label}  dir={direction}  "
              f"component={velocity_component}  bins with data: {(particle_count>0).sum()}/{n_bins}")
        return bin_centers, velocity_profile, velocity_std, particle_count
 
    # ─────────────────────────────────────────────────────────────────────────
    # 3. PHI — solid volume fraction (two-stage: voxel count + Gaussian smooth)
    # ─────────────────────────────────────────────────────────────────────────
 
    def get_bulk_bounds(self, box_bounds, particle_diam=0.01):   # FIX: added self
        """Trim one dp from each y-wall; keep x, z full. Returns (3,2) tensor on DEVICE."""
        b = np.asarray(box_bounds, dtype=np.float64)
        bulk = b.copy()
        bulk[1, 0] += particle_diam
        bulk[1, 1] -= particle_diam
        t = torch.from_numpy(bulk).to(DEVICE)
        print(f"[bounds]  y_full=[{b[1,0]:.4f},{b[1,1]:.4f}]  "
              f"y_bulk=[{bulk[1,0]:.4f},{bulk[1,1]:.4f}]  "
              f"Ly_eff={bulk[1,1]-bulk[1,0]:.4f} m")
        return t
 
    def build_particle_arrays(self, snapshot, bulk_type=3):      # FIX: added self
        """
        Returns
        -------
        centers : (Np,3) float64 DEVICE
        radii   : (Np,)  float64 DEVICE
        tree    : cKDTree on CPU
        """
        type_arr = np.asarray(snapshot['type'])
        mask = (type_arr == bulk_type)
 
        centers_np = np.column_stack([
            np.asarray(snapshot['x'], dtype=np.float64)[mask],
            np.asarray(snapshot['y'], dtype=np.float64)[mask],
            np.asarray(snapshot['z'], dtype=np.float64)[mask],
        ])
        radii_np = np.asarray(snapshot['radius'], dtype=np.float64)[mask]
 
        centers = torch.from_numpy(centers_np).to(DEVICE)
        radii = torch.from_numpy(radii_np).to(DEVICE)
        tree = cKDTree(centers_np)
 
        print(f"[build]  Np={len(radii_np)}  r_mean={radii_np.mean():.5f}  r_max={radii_np.max():.5f}")
        return centers, radii, tree
 
    def compute_max_overlap_and_separation(self, centers, radii, tree,   # FIX: added self
                                           n_sample=2000, safety=1.05):
        """Sample-based max overlap → point separation for sub-voxel template."""
        n_total = centers.shape[0]
        n_sample = min(n_sample, n_total)
        idx = torch.randperm(n_total, generator=torch.Generator().manual_seed(42))[:n_sample]
 
        centers_cpu = centers.cpu().numpy()
        r_max_val = radii.max().item()
        max_overlap = torch.tensor(0.0, dtype=centers.dtype, device=DEVICE)
 
        for i in idx.tolist():
            r_i = radii[i]
            nbrs = tree.query_ball_point(centers_cpu[i], r_i.item() + r_max_val)
            nbrs = [j for j in nbrs if j != i]
            if not nbrs:
                continue
            j_idx = torch.tensor(nbrs, dtype=torch.long, device=DEVICE)
            dists = torch.linalg.norm(centers[i] - centers[j_idx], dim=1)
            overlaps = r_i + radii[j_idx] - dists
            best = overlaps.max()
            if best > max_overlap:
                max_overlap = best
 
        lens_radius = max_overlap / 2.0
        separation = safety * torch.max(lens_radius, radii.max() * 0.05)
        print(f"[overlap]  max_overlap={max_overlap.item():.5f} m  sep={separation.item():.5f} m")
        return max_overlap.item(), separation.item()
 
    def make_template_voxel_points(self, dx, dy, dz, separation):   # FIX: added self
        """Sub-voxel uniform sample points in [0,dx]×[0,dy]×[0,dz]. Built once on DEVICE."""
        nx_pt = max(1, int(np.floor(dx / separation)))
        ny_pt = max(1, int(np.floor(dy / separation)))
        nz_pt = max(1, int(np.floor(dz / separation)))
 
        xs = np.linspace(dx / (2*nx_pt), dx - dx/(2*nx_pt), nx_pt)
        ys = np.linspace(dy / (2*ny_pt), dy - dy/(2*ny_pt), ny_pt)
        zs = np.linspace(dz / (2*nz_pt), dz - dz/(2*nz_pt), nz_pt)
 
        XX, YY, ZZ = np.meshgrid(xs, ys, zs, indexing='ij')
        pts_np = np.column_stack([XX.ravel(), YY.ravel(), ZZ.ravel()])
        template_pts = torch.from_numpy(pts_np).to(DEVICE)
        Npt = len(template_pts)
        print(f"[template]  {nx_pt}×{ny_pt}×{nz_pt} = {Npt} pts/voxel  "
              f"voxel=({dx:.4f},{dy:.4f},{dz:.4f})")
        return template_pts, Npt
 
    def _augment_pbc_images(self, centers, radii, bulk_bounds, cutoff):  # FIX: added self
        """
        Add image particles for PBC in x (dim 0) and z (dim 2) only.
        y is wall-bounded — no images needed.
        """
        x_lo, x_hi = bulk_bounds[0, 0].item(), bulk_bounds[0, 1].item()
        z_lo, z_hi = bulk_bounds[2, 0].item(), bulk_bounds[2, 1].item()
        Lx, Lz = x_hi - x_lo, z_hi - z_lo
 
        cx, cz = centers[:, 0], centers[:, 2]
        near_xlo = (cx - x_lo) < cutoff;  near_xhi = (x_hi - cx) < cutoff
        near_zlo = (cz - z_lo) < cutoff;  near_zhi = (z_hi - cz) < cutoff
 
        shifts = [
            ( Lx,   0.0,  near_xlo),
            (-Lx,   0.0,  near_xhi),
            ( 0.0,  Lz,   near_zlo),
            ( 0.0, -Lz,   near_zhi),
            ( Lx,   Lz,   near_xlo & near_zlo),
            ( Lx,  -Lz,   near_xlo & near_zhi),
            (-Lx,   Lz,   near_xhi & near_zlo),
            (-Lx,  -Lz,   near_xhi & near_zhi),
        ]
 
        aug_c = [centers];  aug_r = [radii]
        for shift_x, shift_z, mask in shifts:
            if mask.any():
                c = centers[mask].clone()
                c[:, 0] += shift_x      # FIX: renamed loop vars (were dx,dz — shadowed outer scope risk)
                c[:, 2] += shift_z
                aug_c.append(c);  aug_r.append(radii[mask])
 
        centers_aug = torch.cat(aug_c, dim=0)
        radii_aug = torch.cat(aug_r, dim=0)
        n_img = centers_aug.shape[0] - centers.shape[0]
        print(f"[pbc images]  {n_img} image particles added (total {centers_aug.shape[0]})")
        return centers_aug, radii_aug
 
    def _phi_one_voxel(self, vox_origin, template_pts, centers, centers_cpu,   # FIX: added self
                       radii, tree, r_max, vox_half_diag):
        """OR-test point counting. Candidate lookup on CPU tree; distance math on GPU."""
        pts = vox_origin + template_pts
        vox_ctr = (vox_origin + template_pts.mean(dim=0)).cpu().numpy()
        candidates = tree.query_ball_point(vox_ctr, vox_half_diag + r_max)
 
        if not candidates:
            return 0.0
 
        cand_idx = torch.tensor(candidates, dtype=torch.long, device=DEVICE)
        dists = torch.cdist(pts, centers[cand_idx], p=2)
        inside_any = (dists <= radii[cand_idx]).any(dim=1)
        return inside_any.float().mean().item()
 
    def compute_phi_voxelwise(self, centers, radii, tree, bulk_bounds,   # FIX: added self
                              grid_shape, separation, verbose=True):
        """
        Sweep all voxels, compute φ via OR-test point counting with PBC images.
        Returns phi_voxel (Nx,Ny,Nz) float64 on DEVICE.
        """
        Nx, Ny, Nz = grid_shape
        diffs = bulk_bounds[:, 1] - bulk_bounds[:, 0]
        Lx, Ly, Lz = diffs[0].item(), diffs[1].item(), diffs[2].item()
        dx, dy, dz = Lx/Nx, Ly/Ny, Lz/Nz
 
        vox_half_diag = 0.5 * (dx**2 + dy**2 + dz**2) ** 0.5
        r_max = radii.max().item()
        cutoff = r_max + vox_half_diag
 
        # PBC image augmentation then rebuild tree
        centers_aug, radii_aug = self._augment_pbc_images(   # FIX: self.
            centers, radii, bulk_bounds, cutoff
        )
        centers_cpu = centers_aug.cpu().numpy()
        tree_aug = cKDTree(centers_cpu)
 
        template_pts, _ = self.make_template_voxel_points(dx, dy, dz, separation)  # FIX: self.
 
        phi_voxel = torch.zeros((Nx, Ny, Nz), dtype=torch.float64, device=DEVICE)
        x0, y0, z0 = bulk_bounds[:, 0].tolist()
        total, done = Nx * Ny * Nz, 0
 
        for ix in range(Nx):
            for iy in range(Ny):
                for iz in range(Nz):
                    origin = torch.tensor(
                        [x0 + ix*dx, y0 + iy*dy, z0 + iz*dz],
                        dtype=torch.float64, device=DEVICE,
                    )
                    phi_voxel[ix, iy, iz] = self._phi_one_voxel(   # FIX: self.
                        origin, template_pts,
                        centers_aug, centers_cpu, radii_aug, tree_aug,
                        r_max, vox_half_diag,
                    )
                    done += 1
                    if verbose and done % max(1, total // 20) == 0:
                        filled = phi_voxel[phi_voxel > 0]
                        mean_s = f"phi_mean={filled.mean().item():.4f}" if filled.numel() else "phi_mean=n/a"
                        print(f"[voxel]  {done}/{total} ({100*done//total}%)  {mean_s}")
 
        if verbose:
            print(f"[voxel done]  mean={phi_voxel.mean():.4f}  "
                  f"min={phi_voxel.min():.4f}  max={phi_voxel.max():.4f}")
        return phi_voxel
 
    def gaussian_smooth_phi(self, phi_voxel, bulk_bounds, grid_shape,   # FIX: added self
                            sigma_phys=None, sigma_vox=None, truncate=3.0):
        """
        Separable Gaussian filter on DEVICE.
        x → circular (PBC)   y → reflect (walls)   z → circular (PBC)
        """
        Nx, Ny, Nz = grid_shape
        diffs = bulk_bounds[:, 1] - bulk_bounds[:, 0]
        dx = (diffs[0] / Nx).item()
        dy = (diffs[1] / Ny).item()
        dz = (diffs[2] / Nz).item()
 
        if sigma_vox is not None:
            sx, sy, sz = sigma_vox
        elif sigma_phys is not None:
            sx, sy, sz = sigma_phys/dx, sigma_phys/dy, sigma_phys/dz
        else:
            sx = sy = sz = 2.0
 
        print(f"[smooth]  sigma_vox=({sx:.2f},{sy:.2f},{sz:.2f})  "
              f"voxel=({dx:.4f},{dy:.4f},{dz:.4f})")
 
        def _kernel1d(sigma):
            r = int(truncate * sigma + 0.5)
            x = torch.arange(-r, r+1, dtype=phi_voxel.dtype, device=DEVICE)
            k = torch.exp(-0.5 * (x / sigma) ** 2)
            return k / k.sum()
 
        kx, ky, kz = _kernel1d(sx), _kernel1d(sy), _kernel1d(sz)
        out = phi_voxel.unsqueeze(0).unsqueeze(0)   # (1,1,Nx,Ny,Nz)
 
        px = len(kx) // 2
        out = F.pad(out, (0, 0, 0, 0, px, px), mode='circular')
        out = F.conv3d(out, kx.view(1, 1, -1, 1, 1))
 
        py = len(ky) // 2
        out = F.pad(out, (0, 0, py, py, 0, 0), mode='reflect')
        out = F.conv3d(out, ky.view(1, 1, 1, -1, 1))
 
        pz = len(kz) // 2
        out = F.pad(out, (pz, pz, 0, 0, 0, 0), mode='circular')
        out = F.conv3d(out, kz.view(1, 1, 1, 1, -1))
 
        phi_smooth = out.squeeze()
        print(f"[smooth]  mean={phi_smooth.mean():.4f}  "
              f"min={phi_smooth.min():.4f}  max={phi_smooth.max():.4f}")
        return phi_smooth
 
    def compute_phi_two_stage(self, snapshot, grid_shape=(50, 38, 10),   # FIX: added self
                              bulk_type=3, particle_diam=0.01,
                              n_overlap_sample=2000, safety=1.05,
                              sigma_phys=None, sigma_vox=None, verbose=True):
        """
        Single-snapshot φ pipeline: build particles → voxel sweep → Gaussian smooth.
 
        Returns
        -------
        phi_voxel  : (Nx,Ny,Nz)       float64  DEVICE
        phi_smooth : (Nx,Ny,Nz)       float64  DEVICE
        phi_tensor : (1,Nx,Ny,Nz,1)   float32  DEVICE  — ML-ready
        """
        centers, radii, tree = self.build_particle_arrays(snapshot, bulk_type)       # FIX: self.
        bulk_bounds = self.get_bulk_bounds(snapshot['box_bounds'], particle_diam)     # FIX: self.
        _, separation = self.compute_max_overlap_and_separation(                      # FIX: self.
            centers, radii, tree, n_sample=n_overlap_sample, safety=safety
        )
        phi_voxel = self.compute_phi_voxelwise(                                       # FIX: self.
            centers, radii, tree, bulk_bounds, grid_shape, separation, verbose
        )
        phi_smooth = self.gaussian_smooth_phi(                                        # FIX: self.
            phi_voxel, bulk_bounds, grid_shape, sigma_phys, sigma_vox
        )
        phi_tensor = phi_smooth.float().unsqueeze(0).unsqueeze(-1)  # (1,Nx,Ny,Nz,1)
        return phi_voxel, phi_smooth, phi_tensor
 
    def compute_phi_timeseries(self, frames, grid_shape=(50, 38, 10),   # FIX: added self
                               bulk_type=3, particle_diam=0.01, safety=1.05,
                               sigma_phys=None, sigma_vox=None, verbose=True):
        """
        Time-series φ pipeline.
 
        Returns
        -------
        phi_voxel_ts  : (Nt,Nx,Ny,Nz)    float32  DEVICE
        phi_smooth_ts : (Nt,Nx,Ny,Nz,1)  float32  DEVICE
        """
        Nt = len(frames)
        Nx, Ny, Nz = grid_shape
 
        # Pre-compute geometry from frame 0 (radii & box are time-invariant)
        cen0, rad0, tree0 = self.build_particle_arrays(frames[0], bulk_type)          # FIX: self.
        bulk_bounds = self.get_bulk_bounds(frames[0]['box_bounds'], particle_diam)     # FIX: self.
        _, separation = self.compute_max_overlap_and_separation(                       # FIX: self.
            cen0, rad0, tree0, n_sample=2000, safety=safety
        )
        diffs = bulk_bounds[:, 1] - bulk_bounds[:, 0]
        dx, dy, dz = (diffs[0]/Nx).item(), (diffs[1]/Ny).item(), (diffs[2]/Nz).item()
        _, Npt = self.make_template_voxel_points(dx, dy, dz, separation)              # FIX: self.
        print(f"\n[timeseries]  Nt={Nt}  grid={Nx}×{Ny}×{Nz}  "
              f"Ly_eff={diffs[1].item():.4f} m  Npt/voxel={Npt}  device={DEVICE}")
 
        phi_v_all = torch.zeros((Nt, Nx, Ny, Nz), dtype=torch.float32, device=DEVICE)
        phi_s_all = torch.zeros((Nt, Nx, Ny, Nz), dtype=torch.float32, device=DEVICE)
 
        for t, frame in enumerate(frames):
            print(f"\n[frame {t+1}/{Nt}]  timestep={frame['timestep']}")
            cen, rad, tr = self.build_particle_arrays(frame, bulk_type)               # FIX: self.
            phi_v = self.compute_phi_voxelwise(                                       # FIX: self.
                cen, rad, tr, bulk_bounds, grid_shape, separation, verbose=False
            )
            phi_s = self.gaussian_smooth_phi(                                         # FIX: self.
                phi_v, bulk_bounds, grid_shape, sigma_phys, sigma_vox
            )
            phi_v_all[t] = phi_v.float()
            phi_s_all[t] = phi_s.float()
            print(f"  raw:    mean={phi_v.mean():.4f}  min={phi_v.min():.4f}  max={phi_v.max():.4f}")
            print(f"  smooth: mean={phi_s.mean():.4f}  min={phi_s.min():.4f}  max={phi_s.max():.4f}")
 
        phi_smooth_ts = phi_s_all.unsqueeze(-1)
        print(f"\n[done]  phi_smooth_ts: {list(phi_smooth_ts.shape)}  device={phi_smooth_ts.device}")
        return phi_v_all, phi_smooth_ts
 
    def diagnose_phi(self, phi_voxel, phi_smooth, bulk_bounds):    # FIX: added self
        """Print φ(y) profile averaged over t, x, z."""
        if phi_smooth.dim() == 5:
            phi_smooth = phi_smooth.squeeze(-1)
 
        Ny = phi_voxel.shape[2]
        Ly = (bulk_bounds[1, 1] - bulk_bounds[1, 0]).item()
        dy = Ly / Ny
        y_lo = bulk_bounds[1, 0].item()
        y_c = [y_lo + (iy + 0.5) * dy for iy in range(Ny)]
 
        phi_v_y = phi_voxel.float().mean(dim=(0, 1, 3)).cpu()
        phi_s_y = phi_smooth.float().mean(dim=(0, 1, 3)).cpu()
 
        print("\n── φ(y) bulk profile  [avg over t, x, z] ───────────────────────")
        print(f"  {'y_centre':>9}  {'raw':>7}  {'smooth':>7}  bar")
        for iy in range(Ny):
            bar = '█' * int(phi_s_y[iy].item() * 40)
            print(f"  y={y_c[iy]:.4f}  {phi_v_y[iy]:.4f}  {phi_s_y[iy]:.4f}  {bar}")
        print("─────────────────────────────────────────────────────────────────")
 

    # ─────────────────────────────────────────────────────────────────────────
    # 4. STRAIN & FABRIC  (contact-based Gaussian coarse-graining)
    # ─────────────────────────────────────────────────────────────────────────

    def compute_strain_and_fabric(
        self,
        frames_sorted,
        grid_shape    = (50, 38, 10),
        x_range       = (0.00, 0.50),
        y_range       = (0.01, 0.39),
        z_range       = (0.00, 0.10),
        xi_mode       = 'particle',   # 'particle' → w=2dp,  'smooth' → w=3dp
        device        = None,
    ):
        """
        Contact-based Gaussian coarse-graining of overlap strain and fabric tensor.

        For each contact c with midpoint m_c, its contribution is spread to ALL
        cell centres within cutoff r_cut = 3w, weighted by φ(|x_cell − m_c|, w).

        Gaussian coarse-graining replaces hard voxel assignment:
            BEFORE:  contact c → cell k if midpoint in cell k   (weight = 1 or 0)
            AFTER:   contact c → all cells k within r_cut        (weight = φ(dist, w))

        Kernel bandwidth w is taken from self.w if already set (e.g. by a prior
        call to estimate_coarse_graining_width or coarse_grain_stress); otherwise
        it is estimated from the first frame via estimate_coarse_graining_width.

        PBC:  x and z periodic (9-image augmentation).
              y wall-bounded  (midpoints outside y_range are discarded).

        Parameters
        ----------
        frames_sorted : list of frame dicts (sorted by timestep)
        grid_shape    : (Nx, Ny, Nz)
        x_range       : (x_lo, x_hi)  — bulk x extent
        y_range       : (y_lo, y_hi)  — bulk y extent (wall-trimmed)
        z_range       : (z_lo, z_hi)  — bulk z extent
        xi_mode       : 'particle' or 'smooth'
        device        : torch device string/object, or None → uses module DEVICE

        Returns
        -------
        strain : torch.Tensor  [N_frames, Nx, Ny, Nz, 1]       float32
            Per-cell weighted mean normalised overlap ε = δ / (r_i + r_j)
        fabric : torch.Tensor  [N_frames, Nx, Ny, Nz, 3, 3]    float32
            Per-cell weighted mean contact-normal dyadic n⊗n
        """
        if device is None:
            device = DEVICE

        Nx, Ny, Nz = grid_shape
        N_frames   = len(frames_sorted)
        N_cells    = Nx * Ny * Nz

        Lx = x_range[1] - x_range[0]
        Lz = z_range[1] - z_range[0]

        # ── Cell centres ──────────────────────────────────────────────────
        xc = 0.5 * (np.linspace(*x_range, Nx + 1)[:-1] + np.linspace(*x_range, Nx + 1)[1:])
        yc = 0.5 * (np.linspace(*y_range, Ny + 1)[:-1] + np.linspace(*y_range, Ny + 1)[1:])
        zc = 0.5 * (np.linspace(*z_range, Nz + 1)[:-1] + np.linspace(*z_range, Nz + 1)[1:])

        Xc, Yc, Zc  = np.meshgrid(xc, yc, zc, indexing='ij')   # each (Nx, Ny, Nz)
        cell_centres = np.column_stack([Xc.ravel(), Yc.ravel(), Zc.ravel()])  # (N_cells, 3)

        # 9 periodic image shifts in x-z (for particle cloud augmentation)
        xz_shifts = np.array(
            [[dx * Lx, 0.0, dz * Lz] for dx in (-1, 0, 1) for dz in (-1, 0, 1)],
            dtype=np.float64,
        )  # (9, 3)

        strain_frames = []
        fabric_frames = []

        for t, frame in enumerate(frames_sorted):

            pos   = np.column_stack([frame['x'], frame['y'], frame['z']])
            radii = np.asarray(frame['radius'], dtype=np.float64)
            N     = len(pos)
            r_max = radii.max()

            # ── Kernel bandwidth ─────────────────────────────────────────
            # Re-use self.w if already estimated; otherwise compute now.
            if self.w is None:
                self.w = self.estimate_coarse_graining_width(frame, xi_mode=xi_mode)
            w     = self.w
            r_cut = self.support_fac * w   # Gaussian negligible beyond support_fac·w

            # ── 9-image augmented particle cloud (PBC in x, z) ───────────
            aug_pos = np.vstack([pos + s for s in xz_shifts])   # (9N, 3)
            img_id  = np.tile(np.arange(N), 9)                   # original particle index

            # ── Contact detection ─────────────────────────────────────────
            tree  = cKDTree(aug_pos)
            pairs = tree.query_pairs(r=2.0 * r_max, output_type='ndarray')

            if len(pairs) == 0:
                print(f"  [{t+1:4d}/{N_frames}]  WARNING: zero pairs found")
                strain_frames.append(np.zeros((Nx, Ny, Nz), dtype=np.float32))
                fabric_frames.append(np.zeros((Nx, Ny, Nz, 3, 3), dtype=np.float32))
                continue

            ai = pairs[:, 0];  aj = pairs[:, 1]
            oi = img_id[ai];   oj = img_id[aj]

            # Drop self-image pairs (same physical particle)
            mask   = oi != oj
            ai, aj = ai[mask], aj[mask]
            oi, oj = oi[mask], oj[mask]

            # Canonicalise so each physical pair appears exactly once
            swap              = oi > oj
            oi[swap], oj[swap] = oj[swap].copy(), oi[swap].copy()
            ai[swap], aj[swap] = aj[swap].copy(), ai[swap].copy()
            _, uniq           = np.unique(np.column_stack([oi, oj]), axis=0, return_index=True)
            ai, aj = ai[uniq], aj[uniq]
            oi, oj = oi[uniq], oj[uniq]

            # ── Branch vector, overlap, unit normal ───────────────────────
            branch = aug_pos[aj] - aug_pos[ai]           # (M, 3)
            dist   = np.linalg.norm(branch, axis=1)      # (M,)
            r_sum  = radii[oi] + radii[oj]               # (M,)
            delta  = r_sum - dist

            in_contact = delta > 0.0
            if not in_contact.any():
                print(f"  [{t+1:4d}/{N_frames}]  WARNING: no real contacts")
                strain_frames.append(np.zeros((Nx, Ny, Nz), dtype=np.float32))
                fabric_frames.append(np.zeros((Nx, Ny, Nz, 3, 3), dtype=np.float32))
                continue

            branch_c = branch[in_contact]
            dist_c   = dist[in_contact]
            delta_c  = delta[in_contact]
            r_sum_c  = r_sum[in_contact]
            ai_c     = ai[in_contact]
            aj_c     = aj[in_contact]

            n   = (branch_c / dist_c[:, None]).astype(np.float32)   # (Mc, 3)
            eps = (delta_c  / r_sum_c).astype(np.float32)           # (Mc,)  normalised overlap
            nnT = n[:, :, None] * n[:, None, :]                     # (Mc, 3, 3)

            # ── Contact midpoints with PBC wrap back into primary cell ────
            mid = 0.5 * (aug_pos[ai_c] + aug_pos[aj_c])             # (Mc, 3)
            mid[:, 0] = (mid[:, 0] - x_range[0]) % Lx + x_range[0]
            mid[:, 2] = (mid[:, 2] - z_range[0]) % Lz + z_range[0]

            # Discard midpoints outside the bulk y range
            valid = (mid[:, 1] >= y_range[0]) & (mid[:, 1] < y_range[1])
            mid_v = mid[valid]
            eps_v = eps[valid]
            nnT_v = nnT[valid]

            if len(eps_v) == 0:
                print(f"  [{t+1:4d}/{N_frames}]  no valid contacts in bulk domain")
                strain_frames.append(np.zeros((Nx, Ny, Nz), dtype=np.float32))
                fabric_frames.append(np.zeros((Nx, Ny, Nz, 3, 3), dtype=np.float32))
                continue

            # ── Gaussian coarse-graining over cell centres ────────────────
            # Augment cell centres with x-z periodic images for PBC-safe
            # radius search, then map hits back to primary cell index.
            aug_cc     = np.vstack([cell_centres + s for s in xz_shifts])  # (9*N_cells, 3)
            aug_cc_idx = np.tile(np.arange(N_cells), 9)                    # primary cell index

            cc_tree = cKDTree(aug_cc)

            # Weighted accumulators (all float64 to avoid precision loss during summation)
            W_eps = np.zeros(N_cells, dtype=np.float64)          # Σ φ·ε
            W_fab = np.zeros((N_cells, 3, 3), dtype=np.float64)  # Σ φ·n⊗n
            W_sum = np.zeros(N_cells, dtype=np.float64)          # Σ φ  (normaliser)

            # For each contact midpoint, find all cell centres within r_cut
            hits = cc_tree.query_ball_point(mid_v, r=r_cut)      # list[list[int]]

            for k, nbr_list in enumerate(hits):
                if not nbr_list:
                    continue
                nbr      = np.asarray(nbr_list, dtype=np.int64)
                cell_idx = aug_cc_idx[nbr]

                dr  = aug_cc[nbr] - mid_v[k]                     # (K, 3)
                phi = self.gaussian_kernel(np.sqrt((dr ** 2).sum(axis=1)), w)  # (K,)

                np.add.at(W_eps, cell_idx, phi * eps_v[k])
                np.add.at(W_sum, cell_idx, phi)
                for a in range(3):
                    for b in range(3):
                        np.add.at(W_fab[:, a, b], cell_idx, phi * nnT_v[k, a, b])

            # ── Normalise ─────────────────────────────────────────────────
            has = W_sum > 0.0

            strain_field = np.zeros(N_cells, dtype=np.float32)
            fabric_field = np.zeros((N_cells, 3, 3), dtype=np.float32)

            strain_field[has] = (W_eps[has] / W_sum[has]).astype(np.float32)
            fabric_field[has] = (W_fab[has] / W_sum[has, None, None]).astype(np.float32)

            strain_frames.append(strain_field.reshape(Nx, Ny, Nz))
            fabric_frames.append(fabric_field.reshape(Nx, Ny, Nz, 3, 3))

            # Cleanup augmented arrays (can be large)
            del aug_pos, pairs, branch, branch_c, n, nnT, nnT_v, W_eps, W_fab, W_sum

            # ── Progress log ──────────────────────────────────────────────
            log_every = max(1, N_frames // 10)
            if (t + 1) % log_every == 0 or t == 0:
                if has.any():
                    mean_aniso = np.sqrt(
                        (fabric_frames[-1].reshape(N_cells, 3, 3)[has] ** 2)
                        .sum(axis=(1, 2))
                    ).mean()
                else:
                    mean_aniso = 0.0
                print(
                    f"  [{t+1:4d}/{N_frames}]"
                    f"  w={w:.4f} m  r_cut={r_cut:.4f} m"
                    f"  contacts={in_contact.sum():8d}"
                    f"  in_domain={valid.sum():8d}"
                    f"  cells_with_weight={has.sum():5d}/{N_cells}"
                    f"  mean_eps={eps_v.mean():.4e}"
                    f"  mean_|A|={mean_aniso:.4e}"
                )

        # ── Stack and move to device ───────────────────────────────────────
        print("\nStacking strain and fabric frames...")

        strain = torch.from_numpy(
            np.stack(strain_frames, axis=0)
        ).unsqueeze(-1).to(device)                    # [T, Nx, Ny, Nz, 1]

        fabric = torch.from_numpy(
            np.stack(fabric_frames, axis=0)
        ).to(device)                                  # [T, Nx, Ny, Nz, 3, 3]

        print(f"  strain : {list(strain.shape)}  {strain.dtype}")
        print(f"  fabric : {list(fabric.shape)}  {fabric.dtype}")

        return strain, fabric 
    # ─────────────────────────────────────────────────────────────────────────
    # TEMPORAL COARSE-GRAINING (strain-based Gaussian window)
    # ─────────────────────────────────────────────────────────────────────────
 
    def strain_based_coarse_graining(self, required_tensor, strain_rate, dump_freq,
                                     time_step, strain_half_window=0.5, overlap_fraction=0.5):
        """
        Temporally smooth any field tensor with a Gaussian window in strain space.
 
        Parameters
        ----------
        required_tensor : ndarray (N_dumps, N_grid, 3, 3)  or any (N_dumps, ...)
        strain_rate     : float [1/s]
        dump_freq       : int   [timesteps per dump]
        time_step       : float [s]
        strain_half_window : float  half-width in strain units (default 0.5)
        overlap_fraction   : float  window overlap in [0,1) (default 0.5)
 
        Returns
        -------
        time_CG    : ndarray (N_out, ...)
        out_strains: ndarray (N_out,)
        """
        N_dumps = required_tensor.shape[0]
        dt_dump = dump_freq * time_step
        strain_per_dump = dt_dump * strain_rate
        total_strain = N_dumps * strain_per_dump
 
        sigma_strain = strain_half_window / 2.0
        stride_strain = strain_half_window * (1.0 - overlap_fraction)
 
        print(f"[temporal CG]  dt_dump={dt_dump:.4f} s  Δγ/dump={strain_per_dump:.4f}  "
              f"γ_total={total_strain:.2f}  σ_γ={sigma_strain:.3f}  stride={stride_strain:.3f}  "
              f"frames/window={strain_half_window/strain_per_dump:.1f}")
 
        out_strains = np.arange(strain_half_window,
                                total_strain - strain_half_window + stride_strain,
                                stride_strain)
        print(f"  N output frames = {len(out_strains)}")
 
        time_CG = np.zeros((len(out_strains),) + required_tensor.shape[1:], dtype=np.float64)
        dump_strains = np.arange(N_dumps) * strain_per_dump
 
        for k, gamma_centre in enumerate(out_strains):
            delta_gamma = dump_strains - gamma_centre
            weights = np.exp(-0.5 * (delta_gamma / sigma_strain) ** 2)
            weights[np.abs(delta_gamma) > 3.0 * sigma_strain] = 0.0
            w_sum = weights.sum()
 
            if w_sum < 1e-12:
                nearest = np.clip(int(np.round(gamma_centre / strain_per_dump)), 0, N_dumps - 1)
                time_CG[k] = required_tensor[nearest]
                continue
 
            weights /= w_sum
            time_CG[k] = np.tensordot(weights, required_tensor, axes=(0, 0))
 
        print("[temporal CG]  done")
        return time_CG, out_strains
 
    # ─────────────────────────────────────────────────────────────────────────
    # I/O
    # ─────────────────────────────────────────────────────────────────────────
 
    def save_tensor(self, frame, stress_field, grid_points, grid_shape,
                    output_folder, filename=None):
        output_folder = Path(output_folder)
        output_folder.mkdir(parents=True, exist_ok=True)
        if filename is None:
            filename = f"stress_tensor_t{frame['timestep']}.pt"
        filepath = output_folder / filename
        data = {
            'stress_tensor': torch.from_numpy(stress_field.astype(np.float32)),
            'grid_points':   torch.from_numpy(grid_points.astype(np.float32)),
            'grid_shape':    grid_shape,
            'box_bounds':    torch.from_numpy(frame['box_bounds'].astype(np.float32)),
            'timestep':      frame['timestep'],
            'N_particles':   frame['N'],
            'cg_width_w':    self.w,
        }
        torch.save(data, filepath)
        print(f"✓ saved {filepath}  shape={stress_field.shape}")
        return filepath
 
    def load_tensor(self, filepath):
        data = torch.load(filepath)
        print(f"✓ loaded {filepath}  shape={data['stress_tensor'].shape}")
        return data

[CG] Using device: cuda


### Main Code Lines (Exemplar Version)

In [3]:
# Objects of the classes defined above:
# =========================================================================================================== #
# 1. File Reading:
folder_path = r"F:\DEM_DATA\const_V_3D\578_085\pour"
# ---------------------------------------------------------------------- 
file_reader = FileReader()
frames = file_reader.read_all_frames(folder_path = folder_path, pattern='dump.stress.*', max_frames=1000000000)
for i in range(len(frames)):
    radius = file_reader.read_radius_from_config(config_file = r"F:\DEM_DATA\const_V_3D\578_085\config_578.txt")
    frames[i]['radius'] = radius
frames_sorted = sorted(frames, key=lambda x: int(x['timestep']))
# =========================================================================================================== #

Reading from F:\DEM_DATA\const_V_3D\578_085\pour
Done! total Frames: 201 dump files


In [ ]:
# 2. Coarse graining....
def main():

    Nf  = len(frames_sorted)
    reference_frame = frames_sorted[0]
 
    # ------------------------------------------------------------------
    # 2. Initialise CG object
    # ------------------------------------------------------------------
    cg  = CompleteCoarseGraining()
    w   = cg.estimate_coarse_graining_width(reference_frame, xi_mode='particle')
    cg.w = w
 
    # Shared evaluation grid (stress + velocity use the same flat grid_points)
    grid_points, grid_shape_3d = cg.create_grid(reference_frame, xi_mode='particle')
    Ngrid = len(grid_points)
    print(f"[grid]  {Ngrid} evaluation points  shape={grid_shape_3d}\n")
 
    # ------------------------------------------------------------------
    # 3. STRESS  (IKH)
    # ------------------------------------------------------------------
    print("=" * 68)
    print("  STRESS  (Irving-Kirkwood-Hardy)")
    print("=" * 68)
 
    stress_list = []
    for t, frame in enumerate(frames_sorted):
        print(f"\n  frame {t+1}/{Nf}  ts={frame['timestep']}")
        sf = cg.coarse_grain_stress(frame, grid_points, w=w, xi_mode='particle')
        stress_list.append(sf.astype(np.float32))
 
    stress_ts = np.stack(stress_list, axis=0)       # (Nf, Ngrid, 3, 3)
    print(f"\n  raw stress time-series: {stress_ts.shape}")
 
    out_strains = None
    
    print("  temporal CG ...")
    vt_match = re.search(r'vt(\d+)', folder_path)
    vt_value = vt_match.group(1) if vt_match else '0'
    
    # ---------- Add values of top wall speed as perr the need and compute their corresponding strain rate values to store them as key-value pair in the dictionary --
    vt_to_strain_rate = {'014': 0.035, '003': 0.00775, '03': 0.0775,'085': 0.2125, '14': 0.35,}
    # ----------------------------------------------------------------------------------------------------------------------------------------------------------------
    strain_rate = vt_to_strain_rate[vt_value]
    dump_freq  = 1e5
    time_step  = 5.18e-6
    stress_cg, out_strains = cg.strain_based_coarse_graining(
        stress_ts, strain_rate, dump_freq, time_step,
        strain_half_window = 0.5,
        overlap_fraction = 0.5,
    )
    
    torch.save_pt({
        'stress_cg':   stress_cg,          # (N_out, Ngrid, 3, 3) or (Nf, Ngrid, 3, 3)
        'stress_raw':  stress_ts,          # (Nf, Ngrid, 3, 3)
        'out_strains': out_strains if out_strains is not None else np.array([]),
        'grid_points': grid_points,        # (Ngrid, 3)
        'grid_shape':  np.array(grid_shape_3d),
        'box_bounds':  reference_frame['box_bounds'],
        'cg_width_w':  np.float32(w),
    }, 'stress_coarse_grained.pt')
 
    # ------------------------------------------------------------------
    # 4. VELOCITY / DEFORMATION RATE  (Goldhirsch-Weinhart)
    # ------------------------------------------------------------------
    print("\n" + "=" * 68)
    print("  VELOCITY  (Goldhirsch-Weinhart, finite-difference)")
    print("=" * 68)
 
    cg.compute_finite_difference_velocities(frames)
 
    vel_list = []
    for t, frame in enumerate(frames):
        print(f"\n  frame {t+1}/{Nf}  ts={frame['timestep']}")
        vf, _ = cg.coarse_grain_velocity(frame, grid_points, w=w, use_fd_velocity=True)
        vel_list.append(vf.astype(np.float32))
 
    vel_ts = np.stack(vel_list, axis=0)             # (Nf, Ngrid, 3)
    print(f"\n  raw velocity time-series: {vel_ts.shape}")
 

    print("  temporal CG ...")
    vel_cg, out_strains = cg.strain_based_coarse_graining(
        stress_ts, strain_rate, dump_freq, time_step,
        strain_half_window = 0.5,
        overlap_fraction = 0.5,
    )

 
    torch.save_pt({
        'velocity_cg':  vel_cg,            # (N_out, Ngrid, 3)
        'velocity_raw': vel_ts,            # (Nf, Ngrid, 3)
        'out_strains':  out_strains if out_strains is not None else np.array([]),
        'grid_points':  grid_points,
        'grid_shape':   np.array(grid_shape_3d),
        'box_bounds':   reference_frame['box_bounds'],
        'cg_width_w':   np.float32(w),
    }, 'velocity_coarse_grained.pt')
 
    # ------------------------------------------------------------------
    # 5. PHI  (two-stage: voxel count + Gaussian smooth)
    # ------------------------------------------------------------------
    print("\n" + "=" * 68)
    print("  PHI  (solid volume fraction, two-stage)")
    print("=" * 68)
 
    phi_v_all, phi_s_all = cg.compute_phi_timeseries(
        frames,
        grid_shape=np.array(grid_shape_3d),
        bulk_type=BULK_TYPE,
        particle_diam=diameter,
        verbose=False,
    )
    # phi_v_all : (Nf, Nx, Ny, Nz)    float32  DEVICE
    # phi_s_all : (Nf, Nx, Ny, Nz, 1) float32  DEVICE
 

    Nx, Ny, Nz = np.array(grid_shape_3d)
    # flatten spatial dims for strain_based_coarse_graining then reshape
    phi_np = phi_v_all.cpu().numpy().reshape(Nf, -1).astype(np.float64)
    print("  temporal CG on phi ...")
    phi_tcg_flat, out_strains = cg.strain_based_coarse_graining(
        phi_np, strain_rate, dump_freq, time_step,
        strain_half_window = 0.5,
        overlap_fraction = 0.5,
    )
    phi_cg = phi_tcg_flat.reshape(-1, Nx, Ny, Nz).astype(np.float32)
    
    torch.save_pt({
        'phi_cg':        phi_cg,                          # (N_out, Nx, Ny, Nz)
        'phi_smooth_ts': phi_s_all.squeeze(-1).cpu().numpy(),  # (Nf, Nx, Ny, Nz)
        'phi_raw_ts':    phi_v_all.cpu().numpy(),          # (Nf, Nx, Ny, Nz)
        'out_strains':   out_strains if out_strains is not None else np.array([]),
        'grid_shape':    np.array(grid_shape_3d),
        'box_bounds':    reference_frame['box_bounds'],
        'particle_diam': diameter,
    }, 'phi_coarse_grained.pt')
 
    # ------------------------------------------------------------------
    # 6. Diagnostics
    # ------------------------------------------------------------------
    print("\n" + "=" * 68)
    print("  DIAGNOSTICS")
    print("=" * 68)
    bulk_bounds = cg.get_bulk_bounds(reference_frame['box_bounds'], diameter)
    cg.diagnose_phi(phi_v_all, phi_s_all, bulk_bounds)
 
    print(f"\n{'='*68}")
    print(f"  DONE.")
    print(f"    stress_coarse_grained.pt    stress_cg  (N_out, N_grid, 3, 3)")
    print(f"    velocity_coarse_grained.pt  velocity_cg (N_out, N_grid, 3)")
    print(f"    phi_coarse_grained.pt       phi_cg     (N_out, Nx, Ny, Nz)")
    print(f"{'='*68}\n")
 
 
if __name__ == '__main__':
    main()

[CG width]  R̄=0.004986 m  dp=0.009973 m  ξ=2·dp (particle-scale IKH)=0.019946 m  support=3.0·ξ=0.059838 m
[grid]  auto Δ = dp = 0.009973 m
[grid]  shape=(51, 39, 11)  y=[0.0100,0.3900]  Δy=0.0100  ξ/2=0.0100  ⚠ Δy > ξ/2
[grid]  21879 evaluation points  shape=(51, 39, 11)

  STRESS  (Irving-Kirkwood-Hardy)

  frame 1/201  ts=0


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed